# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing content was longer and younger on average than declining content. Growing pages averaged about 3,180 words and 184 days of age, while declining pages averaged about 2,311 words and 230 days.

My methodology question is:

**How was trend direction defined, and were the word-count and age comparisons adjusted for client, content type, intent, or existing search visibility?**

The finding is useful as a portfolio-level association, but the comparison does not by itself establish that increasing word count or refreshing age causes growth. Client mix, topic type, and existing visibility may explain part of the difference.

### Finding 2 — The Content Performance Curve

The paper reports that content health peaks around 61–90 days, weakens after approximately 270 days, and appears to recover in the 365+ group.

My methodology question is:

**Does the 365+ group contain a selected population of surviving or recently refreshed pages, and was refresh activity separated from age when estimating the lifecycle pattern?**

The paper itself gives a careful interpretation: older content may recover when refreshed, but age alone should not be treated as causing the rebound. A time-aware design following the same pages over time would support this claim more strongly than comparing separate age groups in one snapshot.

In [10]:
import pandas as pd
paper_findings = pd.DataFrame(
    [
        {
            "finding": "Growing content is longer and younger",
            "evidence_type": "Observational portfolio comparison",
            "methodology_question": (
                "Were client, content type, intent, and existing visibility "
                "controlled when comparing growing and declining pages?"
            ),
            "safe_interpretation": (
                "Word count and age were associated with trend direction, "
                "but the comparison does not establish causation."
            ),
        },
        {
            "finding": "Performance peaks at 61-90 days and declines later",
            "evidence_type": "Cross-sectional age-bucket comparison",
            "methodology_question": (
                "Does the 365+ group contain survivor or refresh-selection bias, "
                "and was refresh activity separated from content age?"
            ),
            "safe_interpretation": (
                "Older refreshed pages may perform better, but age alone "
                "does not explain recovery."
            ),
        },
    ]
)

paper_findings

,finding,evidence_type,methodology_question,safe_interpretation
0,Growing content is longer and younger,Observational portfolio comparison,"Were client, content type, intent, and existin...",Word count and age were associated with trend ...
1,Performance peaks at 61-90 days and declines l...,Cross-sectional age-bucket comparison,Does the 365+ group contain survivor or refres...,"Older refreshed pages may perform better, but ..."


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before/after validation design

The “before” version uses a random row split. This is convenient, but rows from the same client can appear in both training and test sets, allowing client-specific patterns to leak across the split.

The “after” version uses a grouped split by `client_id`. Each client appears entirely in either training or testing, so the model is evaluated on unseen clients.

Both versions use the same safe feature set, preprocessing pipeline, Logistic Regression model, random seed, and evaluation metrics.

In [11]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

target_column = "is_declining_label"

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]
y = df[target_column]
groups = df["client_id"]

print("Rows:", len(df))
print("Clients:", groups.nunique())
print("Base rate:", round(y.mean(), 4))

Rows: 30000
Clients: 32
Base rate: 0.5421


In [12]:
# Before: random row split

X_random_train, X_random_test, y_random_train, y_random_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

# After: grouped client split

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_group_train = X.iloc[group_train_idx]
X_group_test = X.iloc[group_test_idx]

y_group_train = y.iloc[group_train_idx]
y_group_test = y.iloc[group_test_idx]

group_train_clients = set(groups.iloc[group_train_idx])
group_test_clients = set(groups.iloc[group_test_idx])

print("Random split train rows:", len(X_random_train))
print("Random split test rows:", len(X_random_test))
print("Grouped split train rows:", len(X_group_train))
print("Grouped split test rows:", len(X_group_test))
print("Grouped client overlap:", len(group_train_clients & group_test_clients))

Random split train rows: 22500
Random split test rows: 7500
Grouped split train rows: 22885
Grouped split test rows: 7115
Grouped client overlap: 0


In [13]:
def make_model():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore"),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


random_model = make_model()
grouped_model = make_model()

random_model.fit(X_random_train, y_random_train)
grouped_model.fit(X_group_train, y_group_train)

print("Both models trained successfully.")

Both models trained successfully.


In [14]:
def evaluate_model(name, model, X_test_part, y_test_part):
    probabilities = model.predict_proba(X_test_part)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    return {
        "validation_design": name,
        "base_rate": y_test_part.mean(),
        "accuracy": accuracy_score(y_test_part, predictions),
        "precision": precision_score(y_test_part, predictions, zero_division=0),
        "recall": recall_score(y_test_part, predictions, zero_division=0),
        "f1": f1_score(y_test_part, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test_part, probabilities),
    }


validation_results = pd.DataFrame(
    [
        evaluate_model(
            "Random row split",
            random_model,
            X_random_test,
            y_random_test,
        ),
        evaluate_model(
            "Grouped client split",
            grouped_model,
            X_group_test,
            y_group_test,
        ),
    ]
)

validation_results.round(4)

,validation_design,base_rate,accuracy,precision,recall,f1,roc_auc
0,Random row split,0.5420,0.6420,0.6481,0.7429,0.6923,0.6956
1,Grouped client split,0.5165,0.5619,0.5555,0.7603,0.6419,0.5816


### Validation result

The random row split produced stronger results than the grouped client split.

- Random split F1: 0.6923
- Grouped split F1: 0.6419
- Random split ROC-AUC: 0.6956
- Grouped split ROC-AUC: 0.5816

The drop under grouped validation suggests that some of the random-split performance came from client-specific patterns shared across training and test rows.

The grouped result is the more honest estimate for deployment to unseen clients. I therefore use the grouped-client metrics as the headline result, even though they are lower.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I audited the final Week 5 feature set for three forms of leakage:

1. **Label-derived leakage:**  
   `trend_direction` directly defines the target, and `trend_pct` is used to compute `trend_direction`. Both are excluded.

2. **Proxy leakage from overlapping windows:**  
   The 30-day comparison fields used to create trend were removed:
   - `impressions_last_30d`
   - `clicks_last_30d`
   - `sessions_last_30d`
   - `impressions_prev_30d`
   - `clicks_prev_30d`
   - `sessions_prev_30d`

3. **Decision-derived or identifier leakage:**  
   `content_id` and `client_id` are not used as model features. `client_id` is used only for grouping the split. Existing workflow labels and scores are not used as features.

The remaining model still uses current 90-day aggregate features, so it predicts a current-window decline proxy rather than a truly future decline outcome. This is a limitation of the starter dataset and should not be described as future forecasting.

In [15]:
forbidden_or_suspicious = {
    "label_or_direct_source": [
        "is_declining_label",
        "trend_direction",
        "trend_pct",
    ],
    "overlapping_window_inputs": [
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
    ],
    "identifiers_or_grouping_only": [
        "content_id",
        "client_id",
    ],
    "decision_or_generation_metadata": [
        "provider_used",
        "model_used",
    ],
}

audit_rows = []

for category, columns in forbidden_or_suspicious.items():
    for column in columns:
        audit_rows.append(
            {
                "category": category,
                "column": column,
                "present_in_final_features": column in feature_columns,
                "status": (
                    "FAIL"
                    if column in feature_columns
                    else "PASS"
                ),
            }
        )

leakage_audit = pd.DataFrame(audit_rows)

leakage_audit


,category,column,present_in_final_features,status
0,label_or_direct_source,is_declining_label,False,PASS
1,label_or_direct_source,trend_direction,False,PASS
2,label_or_direct_source,trend_pct,False,PASS
3,overlapping_window_inputs,impressions_last_30d,False,PASS
4,overlapping_window_inputs,clicks_last_30d,False,PASS
5,overlapping_window_inputs,sessions_last_30d,False,PASS
6,overlapping_window_inputs,impressions_prev_30d,False,PASS
7,overlapping_window_inputs,clicks_prev_30d,False,PASS
8,overlapping_window_inputs,sessions_prev_30d,False,PASS
9,identifiers_or_grouping_only,content_id,False,PASS


In [16]:
failed_checks = leakage_audit[
    leakage_audit["status"] == "FAIL"
]

print("Final feature count:", len(feature_columns))
print("Leakage checks failed:", len(failed_checks))
print("All audited forbidden fields excluded:", len(failed_checks) == 0)

Final feature count: 32
Leakage checks failed: 0
All audited forbidden fields excluded: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original bold claim:**

The Logistic Regression model predicts which content pages are declining and should be refreshed.

**Rewritten safe claim:**

On this 30,000-row starter dataset, Logistic Regression identified observed decline labels better than an honest transparent rule baseline under a grouped client holdout split.

The grouped-client model achieved an F1 score of 0.6419 and ROC-AUC of 0.5816. These results suggest that the safe search-performance and engagement features contain limited directional information about the current-window decline label.

The model should be used as decision support for prioritizing manual review, not as proof of future decline or as an automatic refresh decision.

In [17]:
claim_audit = pd.DataFrame(
    [
        {
            "claim_version": "Original",
            "claim": (
                "The model predicts which content pages are declining "
                "and should be refreshed."
            ),
            "risk": (
                "Overstates forecasting ability and implies automatic action."
            ),
        },
        {
            "claim_version": "Rewritten",
            "claim": (
                "Under a grouped client holdout split, Logistic Regression "
                "identified observed decline labels better than an honest "
                "transparent baseline and may support manual review prioritization."
            ),
            "risk": (
                "Carefully framed as observed, directional, and decision-support."
            ),
        },
    ]
)

claim_audit

,claim_version,claim,risk
0,Original,The model predicts which content pages are dec...,Overstates forecasting ability and implies aut...
1,Rewritten,"Under a grouped client holdout split, Logistic...","Carefully framed as observed, directional, and..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.